In [ ]:
# Standard library
import ast
import importlib
import os
import sys
from pathlib import Path

# Project setup
sys.path.insert(0, os.path.abspath(".."))

# Third-party libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tiktoken
from dotenv import load_dotenv
from scipy.stats import pearsonr, spearmanr
from transformers import AutoTokenizer

# Local modules
from src import (
    config,
    data,
    evaluate,
    llm,
    prompts,
    qa,
    rag,
    scoring,
)
from src.notebook_eval import (
    essay_correlation,
    essay_evaluation,
    essay_plots,
    article_recitation_plots,
    judge_correlation
)

# Environment variables
load_dotenv(override=True)

# Pandas configuration
pd.set_option("display.max_columns", None)

# Data loading
gpbam_df = pd.read_json("data/gpbam.json")


X_LIMITS = {
    "quality_vs_answer_length": (0, 22),
    "quality_vs_generation_cost": (0, 9.5),
    "quality_vs_legal_reference_similarity": (0, 26),
}
Y_LIMITS = (0, 50)


In [ ]:
def ew_norag(
    judge_instruction_name="ji2",
    *,
    models=None,                       # defaults to config.MODELS_DEV
    judge_models=None,                 # defaults to config.JUDGE_MODELS_DEV
    gpbam_df=None,                     # defaults to pd.read_json("data/gpbam.json")
    result_csv=None,                   # defaults to ./zubaers_result/.../no_rag_{judge_instruction_name}_result.csv
    allow_overwrite_existing_models=False,
    max_tokens=None,
    require_confirmation=True,         # set False to skip the input() prompt in a notebook
    reload_modules=True,
    verbose_judge=True,
):
    """
    Run the no-RAG essay-writing evaluation end to end: for each model, generate an
    answer to every gpbam task directly (no retrieval / no law context), then score
    it with the judge ensemble. No-RAG counterpart to the law-RAG notebook.

    Answers ARE generated here — there is no source-of-answers CSV. Saves after every
    model, so an interrupted run can resume from `result_csv`.

    Args:
        judge_instruction_name: Key into `prompts.JUDGE_INSTRUCTIONS_BY_NAME` selecting
            the judge instruction actually sent to the judge (e.g. "ji1", "ji2").
        models: Iterable of model names to generate answers with and judge. Defaults to
            `config.MODELS_DEV`. Deduplicated while preserving order.
        judge_models: Model names forming the judge ensemble. Defaults to
            `config.JUDGE_MODELS_DEV`.
        gpbam_df: Pre-loaded tasks DataFrame with `facts` (prompts to answer) and
            `solutions` (reference answers the judge scores against). If None, loaded
            from "data/gpbam.json".
        result_csv: Output CSV path. If None, defaults to
            "./zubaers_result/essay_writing/without_rag/{judge_instruction_name}/
            no_rag_{judge_instruction_name}_result.csv" (relative to the current working
            directory). Both read (as a resume cache) and written; re-saved after every
            model.
        allow_overwrite_existing_models: Controls what happens to a model already present
            in `result_csv` (evaluated in a prior run) — True regenerates and re-judges
            it, replacing its old rows; False (default) skips it, so a re-run resumes an
            interrupted run / adds new models without redoing (and re-paying for) work
            already done.
        max_tokens: Max tokens for each model's answer generation. None uses the
            generator's default.
        require_confirmation: If True, print a path/instruction summary and gate the run
            behind an interactive 'yes' prompt. Set False to run non-interactively.
        reload_modules: If True, `importlib.reload` the src modules (prompts, llm, qa,
            config, scoring, evaluate) first, so in-session edits to them take effect.
        verbose_judge: Passed to `JudgeEnsemble` — if True, print judge progress/details.

    Returns:
        The results DataFrame (also written to `result_csv`).
    """


    if reload_modules:
        for mod in (config, data, evaluate, llm, prompts, qa, rag, scoring):
            importlib.reload(mod)

    # --- Judge setup -----------------------------------------------------
    judge_instruction = prompts.JUDGE_INSTRUCTIONS_BY_NAME[judge_instruction_name]
    judge_model_list = judge_models if judge_models is not None else config.JUDGE_MODELS_DEV
    judges = [
        scoring.Judge(model=model, prompt=prompts.build_judge_user(judge_instruction))
        for model in judge_model_list
    ]
    judge = scoring.JudgeEnsemble(judges, verbose=verbose_judge)

    # --- Data & models ---------------------------------------------------
    if gpbam_df is None:
        gpbam_df = pd.read_json("data/gpbam.json")
    if models is None:
        models = config.MODELS_DEV

    # --- Paths -----------------------------------------------------------
    if result_csv is None:
        result_csv = (
            f"./zubaers_result/essay_writing/without_rag/"
            f"{judge_instruction_name}/no_rag_{judge_instruction_name}_result.csv"
        )

    result_csv = Path(result_csv)


    # --- Safety checkpoint ----------------------------------------------
    print("\n" + "=" * 80)
    print("RESULT CSV PATH CHECK")
    print("=" * 80)
    print(f"Output results path:\n{result_csv}")
    print("=" * 80)
    print(f"Judge instruction ({judge_instruction_name})")
    print(judge_instruction[:])
    print("=" * 80)
    print(f"Allow overwrite existing model rows: {allow_overwrite_existing_models}")
    print("\nBefore continuing, verify:")
    print("1. This is the correct paths for this no-RAG run.")
    print("2. The CSV filename is correct.")
    print("3. The instruction text printed above matches judge_instruction_name "
          "(this is what's actually sent — ignore prompts.py's own JUDGE_USER print, "
          "that's an unrelated default and is not used here).")
    print("=" * 80)

    if require_confirmation:
        confirmation = input('Type "yes" to continue running the script: ').strip().lower()
        if confirmation != "yes":
            raise RuntimeError(
                "Execution stopped. Please set `result_csv` correctly before running again."
            )

    # --- Load / create results -------------------------------------------
    result_csv.parent.mkdir(parents=True, exist_ok=True)

    if result_csv.exists():
        print(f"Loading existing judge-specific no-RAG results from {result_csv}")
        model_study = pd.read_csv(result_csv)
    else:
        print("No existing results found. Starting a fresh results DataFrame.")
        model_study = pd.DataFrame()

    completed_models = (
        set(model_study["model"].dropna().unique()) if "model" in model_study else set()
    )

    # --- Evaluation loop -------------------------------------------------
    for model in dict.fromkeys(models):
        if model in completed_models and not allow_overwrite_existing_models:
            print(f"Skipping {model}: already evaluated.")
            continue

        if model in completed_models:
            print(f"Overwriting {model}: removing existing rows before rerun.")
            model_study = model_study[model_study["model"] != model].copy()

        print(f"Evaluating {model}")

        gen = qa.AnswerGenerator(model=model, max_tokens=max_tokens)

        model_df = evaluate.evaluate_model(
            gen,
            tasks=gpbam_df.facts.values,
            solutions=gpbam_df.solutions.values,
            judge=judge,
            model_name=model,
        )

        model_study = pd.concat([model_study, model_df], ignore_index=True)

        # Save after every model so interrupted runs can resume safely.
        model_study.to_csv(result_csv, index=False)
        completed_models.add(model)

    return model_study

In [ ]:
model_study_df = ew_norag(judge_instruction_name="ji2")

In [ ]:
try:
    model_study_df
except NameError:
    model_study_df = pd.read_csv(
        "/root/work/p25-plexam/experiments/zubaers_result/essay_writing/without_rag/ji2/no_rag_ji2_result.csv"
    )

In [ ]:
# Add legal reference similarity scores to the model_study_df

if "legal_ref_sim" not in model_study_df.columns:

    solutions = gpbam_df["solutions"].values

    legal_ref_scores = []

    for _, row in model_study_df.iterrows():
        answer = row["answer"]
        solution_index = int(row["index"])
        solution = solutions[solution_index]

        score = scoring.legal_ref_similarity(answer, solution)
        legal_ref_scores.append(score)

    model_study_df["legal_ref_sim"] = legal_ref_scores
else:
    print("legal_ref_sim column already exists in model_study_df. Skipping computation.")

In [ ]:
'''
    exclude=["mistral-small-3.1-24b-instruct",
             "EuroLLM-22B-Instruct-2512",
             "Phi-4-mini-instruct",
             "gemma-4-31B-it-FP8-block"] 

These models are excluded in the paper because they are not present in the no-RAG experiment (each model had different issue with the RAG setup, sometime context)
'''


# MAIN: score + legal_ref_sim
main = essay_evaluation.make_summary(model_study_df,
                                     caption=r"\textbf{Essay writing performance (no retrieval).} Each model receives the facts of one of the 81 German public-law state-exam cases in GPBam and writes a complete legal essay (\emph{Gutachten}) in German from its parametric knowledge alone -- no statute text is supplied. Every essay is graded against the official reference solution by an ensemble of three LLM judges (gpt-5-nano, qwen3.6-35b-a3b, DeepSeek-V4-Flash), each awarding 0.0--1.0 for correctness, completeness and legal reasoning; an essay's score is the median over the three judges. \textbf{Score} is the mean of those medians over the 81 cases, as a percentage, with its bootstrap standard error ($_{\pm\mathrm{SE}}$, indicating how precisely the mean is estimated). \textbf{Legal Ref. Sim.} is the Jaccard overlap between the statutory references (law book and section, e.g.\ ``\S~42 VwGO'') cited in the essay and those cited in the reference solution, likewise a mean over the 81 cases with bootstrap SE. Higher is better for both. Models are ordered by Score (ascending).",
                                     label="tab:model_comparison_without_rag")
main

In [ ]:
res_main = essay_correlation.correlate(
    main,

    recitation_csv="zubaers_result/article_recitation/article_recitation.csv",

    # We DO NOT need them as they are not present in no-rag experiment
    exclude=["mistral-small-3.1-24b-instruct",
             "EuroLLM-22B-Instruct-2512",
             "Phi-4-mini-instruct",
             "gemma-4-31B-it-FP8-block"]   # drop failed/unwanted models
)
res_main

In [ ]:
essay_correlation.to_latex(
    res_main,
    label="tab:essay_corr_main",
    caption=r"""\textbf{What predicts essay quality (no retrieval).} Spearman rank
correlation $\rho$ between a model's mean essay \textbf{Score} and each
predictor, taken \emph{across} the $n{=}26$ models (one point per model).
\textbf{Legal Ref.\ Sim.}: overlap of statutory citations with the reference
solution; \textbf{Article Recitation}: verbatim statute-recall accuracy on a
separate task. Brackets are 95\% bootstrap CIs. Both correlate with quality, but
Legal Ref.\ Sim.\ is computed from the same essays \textbf{Score} grades---so
their agreement is partly built in---whereas Article Recitation is measured on an
independent task and is thus the stronger evidence.""",
)

In [ ]:
'''
    exclude=["mistral-small-3.1-24b-instruct",
             "EuroLLM-22B-Instruct-2512",
             "Phi-4-mini-instruct",
             "gemma-4-31B-it-FP8-block"] 

These models are excluded in the paper because they are not present in the no-RAG experiment (each model had different issue with the RAG setup, sometime context)
'''


# EXTENDED: + tokens (in k) + cost
extended = essay_evaluation.make_summary(
    model_study_df,
    show_judge_scores=False,
    show_answer_tokens=True,
    show_judge_tokens=True,
    show_gen_cost=True,
    show_judge_cost=True,
    token_divisor=1000, token_digits=1,
    caption=r"\textbf{Essay writing performance (no retrieval, extended).} Setup as in Table~\ref{tab:model_comparison_without_rag}: each model writes a full legal essay for all 81 GPBam exam cases without any supplied statute text, and a three-judge LLM ensemble scores each essay 0.0--1.0 against the reference solution. \textbf{Score} is the mean over the 81 cases of the per-essay median across judges, as a percentage with bootstrap standard error ($_{\pm\mathrm{SE}}$); the three following columns report each individual judge's own mean score in the same way (the FP8 and API variants of the Qwen judge are one logical judge and are merged). \textbf{Legal Ref. Sim.} is the Jaccard overlap of statutory citations with the reference solution. \textbf{Answer Tokens (k)} is the number of tokens the model generated per essay, in thousands, as mean $\pm$ standard deviation across the 81 cases, and \textbf{Judge Tokens (k)} the same for the tokens generated by the full judge panel per essay. \textbf{Gen.\ Cost} and \textbf{Judge Cost} are total spend in USD over all 81 essays, for answer generation and for the judge panel respectively; blank cells mean the provider reported no cost. Models are ordered by Score (ascending).",
    label="tab:model_comparison_without_rag_extended"
)

extended

In [ ]:
res_ext = essay_correlation.correlate(
    extended,
    recitation_csv="zubaers_result/article_recitation/article_recitation.csv",
    exclude=["mistral-small-3.1-24b-instruct",
             "EuroLLM-22B-Instruct-2512",
             "Phi-4-mini-instruct",
             "gemma-4-31B-it-FP8-block"]   # drop failed/unwanted models
)

In [ ]:
essay_correlation.to_latex(
    res_ext,
    label="tab:essay_corr_ext",
    caption=r"""\textbf{What predicts essay quality (extended, no retrieval).}
Spearman $\rho$ of mean essay \textbf{Score} against every other per-model
metric, across the $n{=}26$ models; brackets are 95\% bootstrap CIs. The
individual judges (\textbf{gpt-5-nano}, \textbf{DeepSeek-V4-Flash},
\textbf{qwen3.6-35b-a3b}) correlate near-perfectly by construction---\textbf{Score}
is their median. Among knowledge signals, \textbf{Article Recitation} is measured
on an independent task and is the stronger evidence, while \textbf{Legal Ref.\ Sim.}
is computed from the same essays \textbf{Score} grades, so its agreement is partly
built in; length and cost (\textbf{Answer/Judge Tokens}, \textbf{Gen./Judge Cost})
correlate only weakly. $n<21$ where cost was unreported. Rows ordered by
$|\rho|$.""",
)


In [ ]:
importlib.reload(essay_plots)



individual_panels = (
    {
        "x": essay_plots.COL_ANSWER_TOKENS,
        "xlabel": "Mean answer length (thousand tokens)",
        "title": "Essay quality vs. response length-(no retrieval)",
        "filename": "quality_vs_answer_length",
    },
    {
        "x": essay_plots.COL_GEN_COST,
        "xlabel": "Total generation cost (USD)",
        "title": "Essay quality vs. generation cost-(no retrieval)",
        "filename": "quality_vs_generation_cost",
    },
    {
        "x": essay_plots.COL_LEGAL_REF_SIM,
        "xlabel": "Legal reference similarity (%)",
        "title": "Essay quality vs. legal-reference similarity-(no retrieval)",
        "filename": "quality_vs_legal_reference_similarity",
    },
)


plot_result = essay_plots.make_plots(
    extended,
    panels=individual_panels,

    show_stats_box=True,
    stats_box_location="lower right",

    exclude_models={
        "mistral-small-3.1-24b-instruct",
        "EuroLLM-22B-Instruct-2512",
        "Phi-4-mini-instruct",
        "gemma-4-31B-it-FP8-block",
    },

    exclude_from_cost_plot={
        "Qwen3.6-35B-A3B-FP8",
        # "qwen3.6-35b-a3b",
    },

    label_models=True,
    avoid_label_overlap=True,
    use_log_scale_for_cost=False,
    remove_nonpositive_costs=True,

    save_figures=False,
    output_directory=(
        "/root/work/p25-plexam/experiments/figures/essay_quality"
    ),
    output_format="pdf",

    individual_figsize=(9, 7),
    combined_figsize=(19, 6.5),
    figure_dpi=200,
    save_dpi=300,

    x_limits=X_LIMITS,
    y_limits=Y_LIMITS,

    make_individual_plots=True,
    make_combined_plot=True,
    show_plots=True,

    combined_title=(
        "Essay quality relationships by model-(no retrieval)"
    ),
)

In [ ]:
importlib.reload(article_recitation_plots)
recitation_plot = article_recitation_plots.make_plot(

    extended,


    exclude_models={
        "mistral-small-3.1-24b-instruct",
        "EuroLLM-22B-Instruct-2512",
        "Phi-4-mini-instruct",
        "gemma-4-31B-it-FP8-block",
    },
    recitation_csv=(
        "zubaers_result/article_recitation/article_recitation.csv"
    ),
    title="Essay quality vs. legal knowledge — no retrieval",
    filename="essay_vs_article_recitation_norag",

    save_figure=False,
    show_plot=True,

    label_models=True,
    avoid_label_overlap=True,

    show_legend=True,
    legend_location="lower right",

    show_stats_box=False,
    stats_box_location="lower left",

    x_limits=(0, 54),
    y_limits=Y_LIMITS,

)

### Judge Analysis

In [ ]:
'''
    exclude=["mistral-small-3.1-24b-instruct",
             "EuroLLM-22B-Instruct-2512",
             "Phi-4-mini-instruct",
             "gemma-4-31B-it-FP8-block"] 

These models are excluded in the paper because they are not present in the no-RAG experiment (each model had different issue with the RAG setup, sometime context)
'''


# EXTENDED: + tokens (in k) + cost
judge_analysis_df = essay_evaluation.make_summary(
    model_study_df,
    show_judge_scores=True,
    show_legal_ref_sim=False,
    show_answer_tokens=False,
    show_judge_tokens=False,
    show_gen_cost=False,
    show_judge_cost=False,
    token_divisor=1000, token_digits=1,
    caption=r"\textbf{Comparison of ensemble and individual judge scores (no retrieval).} Each model writes a legal essay for all 81 GPBam cases without supplied statute text. The three LLM judges evaluate each essay against the reference solution on a 0.0--1.0 scale. \textbf{Score} is the mean across cases of the per-essay median of the three judge scores, whereas the remaining columns report each judge's mean score across cases. Values are scaled to percentages and shown with bootstrap standard errors ($_{\pm\mathrm{SE}}$). The FP8 and API variants of the Qwen judge are merged into one logical judge. Differences in judges' scoring levels motivate using the median as a robust ensemble aggregation. Models are ordered by Score (ascending); higher is better.",
    label="tab:judge_analysis_without_rag"
)

judge_analysis_df

In [ ]:
# Correlation is high, but different distribution taht is why we use median instead of min.

In [ ]:
methods = ["spearman", "pearson", "qwk"]


agreement = judge_correlation.analyze_levels(
    model_study_df,
    methods=methods,
    n_boot=10_000,
    n_permutations=10_000,
    seed=42,
    exclude={
        "mistral-small-3.1-24b-instruct",
        "EuroLLM-22B-Instruct-2512",
        "Phi-4-mini-instruct",
        "gemma-4-31B-it-FP8-block",
    },
)

model_analysis = agreement["model"].copy()
essay_analysis = agreement["essay"].copy()
agreement_table = agreement["table"].copy()

display(
    agreement_table.round({
        "coefficient": 3,
        "ci_low": 3,
        "ci_high": 3,
        "p_value": 4,
    })
)


In [ ]:
table_latex = judge_correlation.agreement_table_to_latex(
    agreement["table"],
    label="tab:judge_agreement_without_rag",
)

In [ ]:
fig, axes = judge_correlation.plot_level_agreement(
    agreement,
    methods=["spearman", "pearson", "qwk"],
    save_path="/root/work/p25-plexam/experiments/zubaers_result/essay_writing/without_rag/ji2/judge-analysis/judge_agreement_norag.pdf",
    dpi=300,
    title="Inter-judge agreement at model and essay levels — no retrieval"
)

In [ ]:
figure_latex = judge_correlation.agreement_figure_to_latex(
    "judge_agreement.pdf",
    label="fig:judge_agreement_without_rag",
)

In [ ]:
###
### Heatmaps of correlation matrices for model-level agreement
###

for method in ["spearman", "pearson", "qwk"]:
    fig, ax = judge_correlation.plot_correlation_heatmap(
        model_analysis["correlation_matrices"][method],
        method=method,          # title text only — see the warning below
        level="model",          # title text only
        figsize=(7, 5),
    )
    # fig.savefig(f"judge_heatmap_{method}_model.pdf", bbox_inches="tight")


In [ ]:
###
### Heatmaps of correlation matrices for essay-level agreement
###

for method in ["spearman", "pearson", "qwk"]:
    fig, ax = judge_correlation.plot_correlation_heatmap(
        essay_analysis["correlation_matrices"][method],
        method=method,          # title text only — see the warning below
        level="model",          # title text only
        figsize=(7, 5),
    )
    # fig.savefig(f"judge_heatmap_{method}_model.pdf", bbox_inches="tight")


In [ ]:
for level, analysis in [("model", model_analysis), ("essay", essay_analysis)]:
    fig, ax = judge_correlation.plot_score_distributions(
        analysis["scores"],
        level=level,            # sets the y-axis label; must match the analysis
        figsize=(9, 5),
    )
    fig.savefig(f"/root/work/p25-plexam/experiments/zubaers_result/essay_writing/without_rag/ji2/judge-analysis/judge_score_distributions_{level}.pdf", bbox_inches="tight")


# Archive

In [ ]:
# Doing the median score and average legal reference similarity score for each model in a simplified way


# Find all judge-score columns
score_cols = [
    col for col in model_study_df.columns
    if "score_Judge" in col
]

# Median judge score for each row
df_with_scores = model_study_df.assign(
    median_judge_score=model_study_df[score_cols].median(
        axis=1,
        skipna=True
    )
)

# Average judge score and average legal-reference similarity per model
model_summary = (
    df_with_scores
    .groupby("model", as_index=False)
    .agg(
        average_score=("median_judge_score", "mean"),
        average_legal_ref_sim=("legal_ref_sim", "mean")
    )
)

# Convert both columns to percentages and round to one decimal place
percentage_cols = [
    "average_score",
    "average_legal_ref_sim"
]

model_summary[percentage_cols] = (
    model_summary[percentage_cols] * 100
).round(1)

# Sort by average score
model_summary = (
    model_summary
    .sort_values("average_score", ascending=True)
    .reset_index(drop=True)
)

model_summary

In [ ]:
model_study_df[
    [col for col in model_study_df.columns if "score_Judge" in col]
    + ["legal_ref_sim", "model"]
]